# Renal planner — train the KiPA22 renal artery/vein model (Kaggle / Colab)

Trains the repository's compact 3D U-Net on **KiPA22** CTA and exports `model.onnx` +
`manifest.json`. The planner loads the model only if its held-out Dice passes the release gate
(artery ≥ 0.70, vein ≥ 0.65); with it, the planner can draw arterial territories automatically.

**KiPA22 terms:** join the challenge, accept its research-only terms, and upload the data as a
**private** Kaggle dataset (never public). Layout:
`case_xxxxx/imaging.nii.gz + segmentation.nii.gz` (see `training/README.md`).

**Kaggle:** GPU on, Internet on. Training spans several sessions: each run stops cleanly
before the 12-hour limit and the next run continues from `last_state.pt`.
Run with **Save Version → Save & Run All** so it continues with the browser closed.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /tmp
!rm -rf renal-ct-explorer && git clone -q --depth 1 -b planning-engine https://github.com/ahmederaky2-cmyk/renal-ct-explorer.git
%cd /tmp/renal-ct-explorer
!pip install -q -r requirements-training.txt

In [ ]:
import os, shutil
DATA = "/kaggle/input/YOUR-PRIVATE-KIPA22-DATASET"   # <- change
OUT = "/kaggle/working/kipa22_vessel_unet_v1" if os.path.isdir("/kaggle") else "/content/drive/MyDrive/kipa22_vessel_unet_v1"
FOLD = 0
EPOCHS = 250

# Session 2, 3, …: attach the previous version's output as input and point to it here.
PREVIOUS = ""   # e.g. "/kaggle/input/<this-notebook>/kipa22_vessel_unet_v1"
os.makedirs(OUT, exist_ok=True)
if PREVIOUS:
    for name in ("last_state.pt", "best_model.pt"):
        if os.path.isfile(f"{PREVIOUS}/{name}"):
            shutil.copy2(f"{PREVIOUS}/{name}", f"{OUT}/{name}")
print(sorted(os.listdir(OUT)))

In [ ]:
# Exit code 75 = paused before the session limit -> save this version, then run again with PREVIOUS set.
!PYTHONPATH=. python training/train_kipa22.py --data {DATA} --output {OUT} \
    --fold {FOLD} --epochs {EPOCHS} --resume --time-limit-hours 11; echo "exit code: $?"

In [ ]:
import json, os, zipfile
manifest_path = f"{OUT}/manifest.json"
if os.path.isfile(manifest_path):
    manifest = json.load(open(manifest_path))
    print(json.dumps({k: manifest[k] for k in ("validated", "validation_metrics", "training_cases")}, indent=2))
    with zipfile.ZipFile(f"{OUT}/../kipa22_vessel_model.zip", "w") as archive:
        archive.write(f"{OUT}/model.onnx", "kipa22_vessel_unet_v1/model.onnx")
        archive.write(manifest_path, "kipa22_vessel_unet_v1/manifest.json")
    print("Download kipa22_vessel_model.zip from the output panel.")
else:
    print("Not finished yet: run the next session with PREVIOUS set.")

## Install on the planning Mac

Unzip into `~/RenalPlanner/models/` and add to `~/RenalPlanner/planner.env`:

```
RENAL_VESSEL_MODEL_DIR=/Users/<you>/RenalPlanner/models/kipa22_vessel_unet_v1
```

Patient cases with an arterial phase then get renal artery/vein labels and arterial
territories automatically. KiPA22 is cropped CTA: check the vessels on your own scans before
relying on them. Training all 5 folds (`FOLD = 0…4`) and comparing them is stronger than one fold.